In [ ]:
# 矢板の水平変位プロファイル
# 端断面は Y_target = 18.0 m、許容ずれ tolerance = 0.01 m。
# 壁中央の最大変位は Y ≈ 28.5 m 付近なので、そこを見るときは許容ずれを広げます。
Y_target = 18.0  # m
tolerance = 0.01  # m


## 手順

1. 上のセルで目標の Y と許容ずれを決めます。
2. 次のセルで、ローカルならノートの隣の `src/` と `data/` を使い、Colab でリポジトリが無いときは zip をアップロードして展開します。
3. 最後のセルで、フォルダごとに 1 枚の図を描き、`figures/` に保存します。採用点が 0 件の段階は描かず、警告します。


In [ ]:
from pathlib import Path
import sys
import zipfile

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def find_root(start: Path, *, search_tree: bool) -> Path | None:
    """src/sheetpile_profile.py があるディレクトリを返す。"""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "sheetpile_profile.py").is_file():
            return candidate
    if search_tree and start.is_dir():
        for path in start.rglob("sheetpile_profile.py"):
            if path.parent.name == "src":
                return path.parent.parent
    return None


if IN_COLAB:
    ROOT = find_root(Path("/content"), search_tree=True) or find_root(Path.cwd(), search_tree=False)
    if ROOT is None:
        print("src/ が見つかりません。このリポジトリの zip をアップロードしてください。")
        from google.colab import files

        uploaded = files.upload()
        for name, content in uploaded.items():
            destination = Path("/content") / name
            destination.write_bytes(content)
            if destination.suffix.lower() == ".zip":
                with zipfile.ZipFile(destination) as archive:
                    archive.extractall("/content")
        ROOT = find_root(Path("/content"), search_tree=True)
        if ROOT is None:
            raise FileNotFoundError(
                "zip を展開しても src/sheetpile_profile.py が見つかりません。"
            )
else:
    ROOT = find_root(Path.cwd(), search_tree=False)
    if ROOT is None:
        raise FileNotFoundError(
            "src/sheetpile_profile.py がノートの隣にありません。リポジトリ直下で開いてください。"
        )

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DATA_LANDSIDE = ROOT / "data" / "landside"
DATA_RIVERFRONT = ROOT / "data" / "riverfront"
FIGURES = ROOT / "figures"
print(f"ROOT = {ROOT}")


In [ ]:
import matplotlib.pyplot as plt
from sheetpile_profile import configure_japanese_font, plot_folder, series_warnings

print(f"フォント: {configure_japanese_font()}")
FIGURES.mkdir(parents=True, exist_ok=True)

for folder, caption in (
    (DATA_LANDSIDE, "陸側"),
    (DATA_RIVERFRONT, "川表"),
):
    output = FIGURES / f"{folder.name}.png"
    _fig, series = plot_folder(
        folder,
        Y_target,
        tolerance,
        output_path=output,
        title=f"{caption}（Y = {Y_target:g} m、許容ずれ {tolerance:g} m）",
    )
    print(caption)
    for item in series:
        print(f"  {item.label}: {item.n_points} 点")
    for message in series_warnings(series):
        print(message)
    print(f"保存: {output}")

plt.show()

if IN_COLAB:
    from google.colab import files

    for folder, _caption in (
        (DATA_LANDSIDE, "陸側"),
        (DATA_RIVERFRONT, "川表"),
    ):
        files.download(str(FIGURES / f"{folder.name}.png"))
